In [21]:
from src.train_model import load_and_split_data, split_by_season

X, y, metadata = load_and_split_data('../data/processed/games_final.csv')
X_train, X_val, X_test, y_train, y_val, y_test, meta_train, meta_val, meta_test = split_by_season(
    X, y, metadata, [22020, 22021, 22022, 22023], [22024], [22025]
)


Train: 4770 matches ([22020, 22021, 22022, 22023])
Val:   1230 matches ([22024])
Test:  1230 matches ([22025])


## Baseline
Home team always win

In [23]:
baseline_accuracy = y_test.mean()
print(f"Baseline (home always wins): {baseline_accuracy:.3f}")

Baseline (home always wins): 0.554


## Logistic Regression

In [27]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.preprocessing import StandardScaler


train_median = X_train.median()
X_train_filled = X_train.fillna(train_median)
X_val_filled = X_val.fillna(train_median)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_filled)
X_val_scaled = scaler.transform(X_val_filled)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

preds = log_reg.predict(X_val_scaled)
probs = log_reg.predict_proba(X_val_scaled)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, preds):.3f}")
print(f"Log loss: {log_loss(y_val, probs):.3f}")

Accuracy: 0.672
Log loss: 0.610


## XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_preds = xgb_model.predict(X_val)
xgb_probs = xgb_model.predict_proba(X_val)[:, 1]

print(f"Accuracy: {accuracy_score(y_val, xgb_preds):.3f}")
print(f"Log loss: {log_loss(y_val, xgb_probs):.3f}")

Accuracy: 0.657
Log loss: 0.624


In [29]:
train_preds = xgb_model.predict(X_train)
print(f"Train accuracy: {accuracy_score(y_train, train_preds):.3f}")
print(f"Val accuracy:   {accuracy_score(y_val, xgb_preds):.3f}")

Train accuracy: 0.760
Val accuracy:   0.657


In [30]:
xgb_model = XGBClassifier(
    n_estimators=1000,
    max_depth=3,
    learning_rate=0.03,
    reg_alpha=1,
    reg_lambda=2,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    early_stopping_rounds=30,
    eval_metric='logloss'
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"Zatrzymano na drzewie: {xgb_model.best_iteration}")

train_preds = xgb_model.predict(X_train)
val_preds = xgb_model.predict(X_val)
val_probs = xgb_model.predict_proba(X_val)[:, 1]

print(f"Train accuracy: {accuracy_score(y_train, train_preds):.3f}")
print(f"Val accuracy:   {accuracy_score(y_val, val_preds):.3f}")
print(f"Val log loss:   {log_loss(y_val, val_probs):.3f}")

Zatrzymano na drzewie: 178
Train accuracy: 0.684
Val accuracy:   0.658
Val log loss:   0.621


In [32]:
import pandas as pd

importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).head(10))

HOME_elo                  0.135634
AWAY_elo                  0.099421
HOME_is_back_to_back      0.046140
AWAY_is_back_to_back      0.042894
HOME_rest_days            0.042535
HOME_rolling_fg3_pct_5    0.040225
AWAY_rest_days            0.040096
AWAY_streak               0.038908
HOME_rolling_tov_5        0.036929
AWAY_rolling_tov_5        0.036655
dtype: float32


In [33]:
xgb_model_simple = XGBClassifier(
    n_estimators=1000, max_depth=2, learning_rate=0.02,
    reg_alpha=2, reg_lambda=3, subsample=0.7, colsample_bytree=0.7,
    random_state=42, early_stopping_rounds=30, eval_metric='logloss'
)

xgb_model_simple.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

print(f"Zatrzymano na drzewie: {xgb_model_simple.best_iteration}")

train_preds = xgb_model_simple.predict(X_train)
val_preds = xgb_model_simple.predict(X_val)
val_probs = xgb_model_simple.predict_proba(X_val)[:, 1]

print(f"Train accuracy: {accuracy_score(y_train, train_preds):.3f}")
print(f"Val accuracy:   {accuracy_score(y_val, val_preds):.3f}")
print(f"Val log loss:   {log_loss(y_val, val_probs):.3f}")

Zatrzymano na drzewie: 379
Train accuracy: 0.663
Val accuracy:   0.656
Val log loss:   0.619
